# Testing on PISA dataset with randomized missing data, all 3 missing data handling options tested on the same random values

1. Importing package and dataset

In [ ]:
!pip install iita_python --upgrade
!git clone https://gist.github.com/717f0147675b0c8ed25e50d583c943bf.git

import numpy as np
import iita_python as iita
import iita_python.fit_metrics as iita_fm
from iita_python.additional_ce import AdditionalCEDataset

from iita_python.utils import read_rp
from random import randint, choice, shuffle

Cloning into '717f0147675b0c8ed25e50d583c943bf'...


2. Testing function

In [54]:
def add_missing_values(dataset, skips, choicePool=[i for i in range(5)]):    
    new_dataset = dataset.copy()

    for _ in range(skips):
        while (True):
            a = choice(choicePool)
            b = randint(0, new_dataset.shape[0] - 1)
            if (not (np.isnan(new_dataset.loc[b, a]) or (np.nansum(new_dataset.to_numpy(), axis=0)[a] == 1) or (np.nansum(new_dataset.to_numpy(), axis=1)[b] == 1))):
                break
        new_dataset.loc[b, a] = np.nan
    
    return new_dataset

In [55]:
def get_all_ces(rp):
    normal_data = iita.Dataset(rp)
    add_ce_data = AdditionalCEDataset(rp)

    return [normal_data.ce, normal_data.relative_ce, add_ce_data.missing_value_substitution_ce()]

In [ ]:
def test(orig_data, skips, calc_count=3, calculator=get_all_ces, bias=[1,1,1,1,1]):
    correct = [True for _ in range(calc_count)]
    any_correct = calc_count
    correct_qo = None
    correct_count = [0 for _ in range(calc_count)]
    missing_amount = 0
    data = orig_data.copy()

    choicePool = []
    items = list(range(data.shape[1]))
    shuffle(items)

    for i, item in enumerate(items):
        for _ in range(bias[i]):
            choicePool.append(item)

    while (any_correct and missing_amount < data.shape[0]*data.shape[1] - 10):
        print(missing_amount, correct)
        ces = calculator(data)

        test_dataset = iita.Dataset(data)

        all_qos = [iita.ind_gen(ce, test_dataset.items) for ce in ces]

        for (i, qos) in enumerate(all_qos):
            if (not correct[i]):
                continue

            best_qo_diff = float('inf')

            for j, qo in enumerate(qos):
                qo_diff = iita_fm.mini_iita_fit(test_dataset, qo)
                if (qo_diff < best_qo_diff):
                    best_qo_diff = qo_diff
                    best_qo_id = j

            best_qo = sorted([(int(a), int(b)) for a, b in all_qos[i][best_qo_id].get_edge_list()])
            if (correct_qo is None):
                correct_qo = best_qo

            if (best_qo != correct_qo):
                correct[i] = False
                any_correct -= 1
            else:
                correct_count[i] += skips

        missing_amount += skips
        data = add_missing_values(data, skips, choicePool)

    return correct_count

In [ ]:
def save_snapshot_full(iter_res, iter_idx):
    for name, results in iter_res.items():
        out_file = f"iter_test_res_{name}_upto_{iter_idx}.csv"
        with open(out_file, "w") as f:
            for r in results:
                f.write(",".join(map(str, r)) + "\n")

def mean_all(results):
    arr = np.array(results)
    return arr.mean(axis=0)

def iter_test_multi(datasets, skips, iters, log_every = 10, save_every = 50, **kwargs):
    """
    datasets: dict {name -> data}
    """
    names = list(datasets.keys())
    iter_res = {name: [] for name in names}

    for i in range(iters):
        print(f'\nITER {i}')

        for name, data in datasets.items():
            print(f'  Dataset: {name}')
            res = test(data, skips, **kwargs)
            print(f'    -> {res}')
            iter_res[name].append(res)

        # every log_every: show cumulative averages
        if (i + 1) % log_every == 0:
            print(f"\n=== Cumulative averages over all {i+1} iterations ===")
            for name, results in iter_res.items():
                means = mean_all(results)
                print(f"{name}:")
                for j, m in enumerate(means):
                    print(f"  impl_{j+1}: {m:.2f}")

        # alle save_every: save full results up to now
        if (i + 1) % save_every == 0:
            save_snapshot_full(iter_res, i + 1)
            print(f"\n[Snapshot (full results) saved up to iteration {i+1}]")
    return iter_res


3. Running the tests

In [ ]:
datasets = {
    'pisa_orig': read_rp('./717f0147675b0c8ed25e50d583c943bf/pisa.csv')
}

iters = 2000 #amount of iterations to do
skips = datasets[0].shape[0] * datasets[0].shape[1] / 100 #amount of missing values to add at a time

res = iter_test_multi(datasets, skips, iters)